## 1. Ingesta de datos
- Cargar con **Pandas** los 4 ficheros: `movies.csv`, `ratings.csv`, `links.csv`, `tags.csv`

In [24]:
import pandas as pd
import os
movies=pd.read_csv(os.path.join('data','movies.csv'))
ratings=pd.read_csv(os.path.join('data','ratings.csv'))
links=pd.read_csv(os.path.join('data','links.csv'))
tags=pd.read_csv(os.path.join('data','tags.csv'))

- Comprobar para cada `DataFrame`: `shape`, columnas, `dtype`, `head` y conteo de nulos

In [25]:
def informacion(datos:pd.DataFrame)->dict:
    return {
    'shape':datos.shape,
    'columns':datos.columns.to_list(),
    'dtype':datos.dtypes.astype(str).to_dict(),
    'nulos': datos[datos.isna().any(axis=1)].shape[0]
    }

In [26]:
movies_info=informacion(movies)
ratings_info=informacion(ratings)
links_info=informacion(links)
tags_info=informacion(tags)

## 2. Columna `year`desde titulo
- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.

In [27]:
import re

In [28]:
def year_from_title(datos:str)->int|None:
    year=re.findall(r'\(\d{4}\)',datos)
    if year:
        return int(year[0][1:5])

- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).

In [29]:
movies['year']=movies['title'].apply(year_from_title)

- Editar el campo `title` para que no contenga el año de estreno.

In [30]:
def delete_year_from_title(datos:str)->str:
    year=re.findall(r'\(\d{4}\)',datos)
    if year:
        return datos[:-6].strip()
    else:
        return datos

In [31]:
movies['title']=movies['title'].apply(delete_year_from_title)

- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [32]:
print(f'sin año reconocible hay {movies[movies["year"].isna()]["movieId"].count()} peliculas')
movies[movies['year'].isna()]

sin año reconocible hay 13 peliculas


,movieId,title,genres,year
6059,40697,Babylon 5,Sci-Fi,NaN
9031,140956,Ready Player One,Action|Sci-Fi|Thriller,NaN
9091,143410,Hyena Road,(no genres listed),NaN
9138,147250,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed),NaN
9179,149334,Nocturnal Animals,Drama|Thriller,NaN
9259,156605,Paterson,(no genres listed),NaN
9367,162414,Moonlight,Drama,NaN
9448,167570,The OA,(no genres listed),NaN
9514,171495,Cosmos,(no genres listed),NaN
9515,171631,Maria Bamford: Old Baby,(no genres listed),NaN


## 3. Unificación de datos merge (*merge/join*)


In [33]:


# Tabla 1: película-usuario-rating
movie_ratings = pd.merge(ratings, movies, on="movieId", how="inner")
print("Película-usuario-rating:")
print(movie_ratings.head(10))

# Tabla 2: película-tags
movie_tags = pd.merge(tags, movies, on="movieId", how="inner")
print("\nPelícula-tags:")
print(movie_tags.head(10))

# Documentación de filas perdidas/multiplicadas
print(f"\nRatings originales: {ratings.shape[0]}")
print(f"Movie_ratings tras merge: {movie_ratings.shape[0]}")
print(f"Tags originales: {tags.shape[0]}")
print(f"Movie_tags tras merge: {movie_tags.shape[0]}")

Película-usuario-rating:
   userId  movieId  rating  timestamp                 title  \
0       1        1     4.0  964982703             Toy Story   
1       1        3     4.0  964981247      Grumpier Old Men   
2       1        6     4.0  964982224                  Heat   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en)   
4       1       50     5.0  964982931   Usual Suspects, The   
5       1       70     3.0  964982400   From Dusk Till Dawn   
6       1      101     5.0  964980868         Bottle Rocket   
7       1      110     4.0  964982176            Braveheart   
8       1      151     5.0  964984041               Rob Roy   
9       1      157     5.0  964984100        Canadian Bacon   

                                        genres    year  
0  Adventure|Animation|Children|Comedy|Fantasy  1995.0  
1                               Comedy|Romance  1995.0  
2                        Action|Crime|Thriller  1995.0  
3                             Mystery|Thriller  1995.

## 4. Agregaciones y segmentación

In [34]:


# Agrupamos por película y sacamos varias estadísticas a la vez (agregación multi-columna)
stats_por_pelicula = movie_ratings.groupby("title").agg(
    media_rating=("rating", "mean"),
    num_valoraciones=("rating", "count"))
print(stats_por_pelicula.head(10))

                                  media_rating  num_valoraciones
title                                                           
'71                                   4.000000                 1
'Hellboy': The Seeds of Creation      4.000000                 1
'Round Midnight                       3.500000                 2
'Salem's Lot                          5.000000                 1
'Til There Was You                    4.000000                 2
'Tis the Season for Love              1.500000                 1
'burbs, The                           3.176471                17
'night Mother                         3.000000                 1
(500) Days of Summer                  3.666667                42
*batteries not included               3.285714                 7


In [35]:
# Comparamos valoraciones por década
movie_ratings["decada"] = (movie_ratings["year"] // 10) * 10

stats_por_decada = movie_ratings.groupby("decada").agg(
    media_rating=("rating", "mean"),
    num_valoraciones=("rating", "count"))
print(stats_por_decada)


        media_rating  num_valoraciones
decada                                
1900.0      3.312500                 8
1910.0      3.312500                 8
1920.0      3.740000               125
1930.0      3.733624               687
1940.0      3.870572              1101
1950.0      3.845011              1784
1960.0      3.808083              2858
1970.0      3.775676              4995
1980.0      3.518355             12912
1990.0      3.434613             37087
2000.0      3.465750             29766
2010.0      3.488774              9487


## 5. Preguntas sobre los datos (movies)

In [36]:
# --- Apartado 5: Preguntas sobre movies ---

# 1. Cuántas películas están listadas
print(f"1. Total de películas: {movies.shape[0]}")

1. Total de películas: 9742


In [37]:
# 2. Cuáles son las más antiguas (menor año extraído)
mas_antiguas = movies[movies["year"] == movies["year"].min()]
print("2. Películas más antiguas:")
print(mas_antiguas[["title", "year"]])

2. Películas más antiguas:
                                              title    year
5868  Trip to the Moon, A (Voyage dans la lune, Le)  1902.0


In [38]:
# 3. Cuántas tienen "Dracula" en el título (sin distinguir mayúsculas)
dracula = movies[movies["title"].str.contains("Dracula", case=False, na=False)]
print(f"3. Películas con 'Dracula': {dracula.shape[0]}")

3. Películas con 'Dracula': 9


In [39]:
# 4. Títulos más comunes
titulos_comunes = movies["title"].value_counts().head(10)
print("4. Títulos más repetidos:")
print(titulos_comunes)

4. Títulos más repetidos:
title
Hamlet                          5
Christmas Carol, A              4
Three Musketeers, The           4
Misérables, Les                 4
Jane Eyre                       4
Emma                            3
Alice in Wonderland             3
Mummy, The                      3
Gulliver's Travels              3
Hunchback of Notre Dame, The    3
Name: count, dtype: int64


In [40]:
# 5. Películas con "Exorcist" ordenadas de más antigua a más moderna
exorcist = movies[movies["title"].str.contains("Exorcist", case=False, na=False)]
exorcist_ordenado = exorcist.sort_values("year")
print("5. Películas 'Exorcist' ordenadas:")
print(exorcist_ordenado[["title", "year"]])

5. Películas 'Exorcist' ordenadas:
                                  title    year
1472                      Exorcist, The  1973.0
1473           Exorcist II: The Heretic  1977.0
1474                  Exorcist III, The  1990.0
5315            Exorcist: The Beginning  2004.0
5904  Dominion: Prequel to the Exorcist  2005.0
9173           Blue Exorcist: The Movie  2012.0


In [41]:
# 6. Cuántas con año 1950
print(f"6. Películas del año 1950: {movies[movies['year'] == 1950].shape[0]}")

6. Películas del año 1950: 21


In [42]:
# 7. Cuántas entre 1950 y 1959 inclusive
entre_50_59 = movies[(movies["year"] >= 1950) & (movies["year"] <= 1959)]
print(f"7. Películas entre 1950 y 1959: {entre_50_59.shape[0]}")

7. Películas entre 1950 y 1959: 279


In [43]:
# 8. Año de la película con título exacto "Batman" (y contraste con otras Batman)
batman_exacto = movies[movies["title"] == "Batman"]
print("8. Batman (título exacto):")
print(batman_exacto[["title", "year"]])

batman_todas = movies[movies["title"].str.contains("Batman", case=False, na=False)]
print("Todas las que contienen 'Batman':")
print(batman_todas[["title", "year"]])

8. Batman (título exacto):
       title    year
509   Batman  1989.0
5463  Batman  1966.0
Todas las que contienen 'Batman':
                                         title    year
126                             Batman Forever  1995.0
509                                     Batman  1989.0
1060                            Batman Returns  1992.0
1174                            Batman & Robin  1997.0
2418              Batman: Mask of the Phantasm  1993.0
5463                                    Batman  1966.0
5620                Batman/Superman Movie, The  1998.0
5631        Batman Beyond: Return of the Joker  2000.0
5917                             Batman Begins  2005.0
6815                     Batman: Gotham Knight  2008.0
7380                Batman: Under the Red Hood  2010.0
7731                          Batman: Year One  2011.0
7903           Superman/Batman: Public Enemies  2009.0
8032   Batman: The Dark Knight Returns, Part 1  2012.0
8080   Batman: The Dark Knight Returns, Part 2  201

In [44]:
# 9. Películas con tag "sci-fi" y "adventure"
scifi_tags = tags[tags["tag"].str.lower() == "sci-fi"]["movieId"]
adventure_tags = tags[tags["tag"].str.lower() == "adventure"]["movieId"]
movies_con_ambos = movies[movies["movieId"].isin(scifi_tags) & movies["movieId"].isin(adventure_tags)]
print("9. Películas con tags 'sci-fi' y 'adventure':")
print(movies_con_ambos[["title"]])

9. Películas con tags 'sci-fi' y 'adventure':
Empty DataFrame
Columns: [title]
Index: []


In [45]:
# 10. Tag más repetida
tag_mas_repetida = tags["tag"].value_counts().idxmax()
print(f"10. Tag más repetida: {tag_mas_repetida}")

10. Tag más repetida: In Netflix queue
